# 06 — Answer Correctness, By Hand

Faithfulness (notebook `05`) asks "is this claim backed by the context." Correctness asks a different question entirely: **is this answer actually right, compared to the real, gold answer** -- regardless of how it got there. An answer can be perfectly faithful to its context and still be incomplete, or answer a narrower question than the one asked. These two metrics catch genuinely different failures, which is exactly why production systems track both instead of picking one.


In [ ]:
import json
import os

DATA_DIR = os.path.join("..", "data")
with open(os.path.join(DATA_DIR, "eval_examples.json")) as f:
    examples = json.load(f)


## Worked example 1 — clean and fully correct

Start with the easy one again, as a calibration point.


In [ ]:
ex = examples[0]
print("System:", ex["system_answer"])
print("Gold:  ", ex["gold_answer"])


**Judgment: fully correct.** Reworded, not shortened, nothing missing, nothing added. This is what "matches the gold answer" should mean -- meaning-level agreement, not string-level.


## Worked example 2 — technically an answer, but the wrong clause

This is the interesting one. The contract has *two* liability-limiting clauses sitting back to back. The gold label points at one of them. The system found the other.


In [ ]:
ex = examples[7]
print("Context:")
print(ex["context_text"])
print()
print("Gold:  ", ex["gold_answer"])
print("System:", ex["system_answer"])


**Judgment: partially correct.** Read the full context: it contains a dollar-figure cap ("limited... to [**] US Dollars ($[**])") *and*, right after it, a separate clause excluding indirect and consequential damages entirely. CUAD's gold label points at the second clause. The system's answer is about the first one. Both clauses are real, both are genuinely about limiting liability, and the system didn't invent anything -- it's faithful to the context, just not to the *specific span* the label happened to point at.

This is worth naming precisely: it's not wrong the way a hallucination is wrong. It found a real, defensible answer to "what limits this contract's liability" -- just not the one gold answer designated as *the* answer, when the contract actually supports more than one. Call this "partially correct" rather than "incorrect": marking it fully wrong would be scoring the system for genuinely finding relevant information, and marking it fully correct would ignore that it missed what was actually being asked for.


## Worked example 3 — when "I don't know" is the fully correct answer

Correctness has a special case that's easy to score wrong by instinct: for a genuinely unanswerable question, the gold answer is "not stated" -- and an honest refusal that matches it should score as **fully correct**, not as a non-answer.


In [ ]:
ex = examples[16]
print("Question:", ex["question"])
print("Gold:  ", ex["gold_answer"])
print("System:", ex["system_answer"])


**Judgment: fully correct.** "The context does not contain an answer to that question" says exactly the same thing as the gold label "Not stated in the documents," just in different words. If your scoring instinct is to only give credit for answers that *state a fact*, you'll systematically punish a system for being honest -- which trains toward exactly the wrong behavior. A refusal that matches a genuinely unanswerable gold label is not a missing answer. It's the correct one.


## Revisiting notebook 02 — now with the full vocabulary

Back in notebook `02`, you guessed root causes with only intuition to go on. Example 13 was flagged there as `ground_truth_questionable`. Look at it again now that you have faithfulness and correctness as separate, named tools.


In [ ]:
ex = examples[13]
print("Question:", ex["question"])
print("Gold:  ", ex["gold_answer"])
print("System:", ex["system_answer"])


Run both checks on it deliberately: is the system's answer **faithful** to the context (does it only claim what the context actually supports)? Almost certainly yes -- both buildings really are described as real estate in the passage. Is it **correct** against the gold label? By strict matching, no -- gold says "no," the system says "yes." A metric that only measures correctness against gold would flag this as a clean failure. A metric that also checks faithfulness would show a faithful answer disagreeing with a label that itself doesn't hold up under the same scrutiny you'd apply to any answer. That gap -- faithful but "incorrect" against a questionable label -- is precisely why tracking both metrics, instead of trusting one gold-comparison number, catches something a single score would have hidden.


## Your turn

Same three-way judgment -- fully correct, partially correct, incorrect -- for the rest. For the unanswerable set (15-19), decide whether "not stated" should count as fully correct for each, the same way it did in worked example 3.


In [ ]:
for i in range(20):
    ex = examples[i]
    print(f"[{i}] System: {ex['system_answer'][:150]}")
    print(f"    Gold:   {ex['gold_answer'][:150]}")
    print()


In [ ]:
correctness_notes = {
    0: "fully_correct",
    7: "partially_correct",
    13: "incorrect_vs_gold_but_gold_is_questionable",
    16: "fully_correct",  # honest refusal matches "not stated" gold
    # Judge the remaining 16:
    # 1: "",
}

print(f"Judged {len(correctness_notes)}/20")


## What we learned

**Terms to keep, in plain English:**

| Term | Plain meaning |
|---|---|
| Correctness | Whether an answer matches the gold answer in meaning, not just in wording |
| Partially correct | A real, defensible answer that doesn't match the specific gold-labeled span |
| "Not stated" as correct | A refusal that should score as fully correct when the gold answer is genuinely "not stated" |

You now have two independent axes instead of one blurry "is this good" judgment: **faithfulness** (is it grounded in the context) and **correctness** (does it match what's actually being asked for). Example 7 was faithful but only partially correct. Example 13 was faithful but scored "incorrect" against a gold label that itself doesn't hold up. Neither axis alone would have shown you both of those things.

**Next up:** you've now done faithfulness and correctness by hand, on all 20 examples, with real reasoning behind every verdict. Notebook `07` automates both judges in code and checks whether they agree with the calls you just made.

**Exercises:**

- Finish `correctness_notes` for the remaining 16 examples.
- Go back through your notebook 02 root-cause guesses one at a time. For each "bad" verdict, decide now whether it was a faithfulness problem, a correctness problem, or both -- were your first-pass guesses closer to one or the other?
- For example 7, would you still call it "partially correct" if the question had been phrased "What is the exact dollar cap on liability?" instead of the generic "Cap On Liability" label? Does a more specific question change how much credit a near-miss deserves?
